In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import OneClassSVM
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder, StandardScaler
import joblib


In [ ]:
# Cell 2: Load CSV data and drop irrelevant columns
df = pd.read_csv('/content/Book4.csv')

# Drop all columns with 'Unnamed' in their name
df_clean = df.loc[:, ~df.columns.str.contains('^Unnamed')]

print("Columns after dropping unnamed:")
print(df_clean.columns.tolist())


Columns after dropping unnamed:
['Type', 'Date', 'Num', 'Memo', 'Name', 'Item', 'Qty', 'Sales Price', 'Amount', 'Balance']


In [ ]:
df_clean = df_clean.dropna()
print(f"Data shape after dropping missing values: {df_clean.shape}")

Data shape after dropping missing values: (132, 10)


In [ ]:
target = 'Amount'  # Confirm this is your target column

if target not in df_clean.columns:
    raise ValueError(f"Target column '{target}' not found in dataset.")

features = df_clean.columns.drop(target)
print(f"Features: {features.tolist()}")


Features: ['Type', 'Date', 'Num', 'Memo', 'Name', 'Item', 'Qty', 'Sales Price', 'Balance']


In [ ]:
categorical_cols = df_clean.select_dtypes(include=['object']).columns
for col in categorical_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col])


In [ ]:
X = df_clean[features]
y = df_clean[target]

print(f"Feature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")


Feature matrix shape: (132, 9)
Target vector shape: (132,)


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42)

print(f"Training data shape: {X_train.shape}")
print(f"Test data shape: {X_test.shape}")


Training data shape: (105, 9)
Test data shape: (27, 9)


In [ ]:
ocsvm = OneClassSVM(nu=0.05, kernel='rbf', gamma=0.1)
ocsvm.fit(X_train)


OneClassSVM(gamma=0.1, nu=0.05)

In [ ]:
anomaly_pred_test = ocsvm.predict(X_test)
print("Anomaly prediction counts on test set:")
print(pd.Series(anomaly_pred_test).value_counts())


Anomaly prediction counts on test set:
 1    15
-1    12
Name: count, dtype: int64


In [ ]:
rf = RandomForestRegressor(random_state=42)

param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [5, 10, None]
}


In [ ]:
grid_search = GridSearchCV(rf, param_grid, cv=3, scoring='neg_mean_squared_error')
grid_search.fit(X_train, y_train)

print("Best parameters found:")
print(grid_search.best_params_)


Best parameters found:
{'max_depth': 5, 'n_estimators': 100}


In [ ]:
best_rf = grid_search.best_estimator_
y_pred = best_rf.predict(X_test)


In [ ]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Test RMSE: {rmse:.4f}")

threshold = 0.1 * y.mean()
print(f"RMSE threshold (10% of mean target): {threshold:.4f}")

if rmse <= threshold:
    print("Model meets quantitative objective.")
else:
    print("Model does NOT meet quantitative objective.")


Test RMSE: 383.3022
RMSE threshold (10% of mean target): 18.0606
Model does NOT meet quantitative objective.


In [ ]:
joblib.dump(best_rf, 'rf_model.joblib')
joblib.dump(scaler, 'scaler.joblib')


['scaler.joblib']

In [ ]:
def predict_and_append(new_record: dict, model, scaler, dataset_path: str) -> dict:
    new_df = pd.DataFrame([new_record])

    # Encode categorical columns - use same LabelEncoders as training if saved
    for col in categorical_cols:
        if col in new_df.columns:
            le = LabelEncoder()
            new_df[col] = le.fit_transform(new_df[col])

    new_X = scaler.transform(new_df[features])
    prediction = model.predict(new_X)[0]

    new_record[target] = prediction
    pd.DataFrame([new_record]).to_csv(dataset_path, mode='a', header=False, index=False)

    return {'prediction': prediction}


In [ ]:
# Replace sample values with actual valid values for features
sample_new_record = {col: 0 for col in features}
result = predict_and_append(sample_new_record, best_rf, scaler, 'Book4_converted.csv')

print(f"Sample prediction: {result['prediction']}")


Sample prediction: 179.2


In [ ]:
import joblib
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

# Load the trained model and scaler
model = joblib.load('rf_model.joblib')  # Replace with your saved model path
scaler = joblib.load('scaler.joblib')  # Replace with your saved scaler path

# Define the feature columns (same as training)
features = ['Qty', 'Sales Price', 'Balance', 'Item', 'Name']  # Modify as per your actual features

# Function to predict sales amount for a new record
def predict_and_append(new_record, model, scaler, dataset_path):
    """
    Predict sales for a new record and append the result to a CSV dataset.

    Parameters:
    new_record (dict): The new transaction data (features in a dictionary form).
    model (trained model): The trained Random Forest model.
    scaler (StandardScaler): The scaler used to preprocess features.
    dataset_path (str): Path to the CSV file to append the new record with prediction.

    Returns:
    dict: Prediction result with 'prediction' key containing the predicted sales amount.
    """
    # Convert the new record to DataFrame
    new_df = pd.DataFrame([new_record])

    # Encode categorical columns if necessary
    for col in new_df.select_dtypes(include=['object']).columns:
        le = LabelEncoder()
        new_df[col] = le.fit_transform(new_df[col])

    # Preprocess features: scale the input data
    new_X = scaler.transform(new_df[features])

    # Get prediction
    prediction = model.predict(new_X)[0]

    # Append the prediction to the record and save it to the CSV file
    new_record['Prediction'] = prediction
    pd.DataFrame([new_record]).to_csv(dataset_path, mode='a', header=False, index=False)

    return {'prediction': prediction}

In [ ]:
# New transaction record (example)
new_transaction = {
    'Qty': 20,
    'Sales Price': 15.0,
    'Balance': 300.0,
    'Item': 'Product A',
    'Name': 'Customer XYZ'
}

# Path to the existing dataset where new records will be appended
dataset_path = 'sales_data.csv'  # Update with your actual file path

# Call the function to predict and append the new record
result = predict_and_append(new_transaction, model, scaler, dataset_path)

# Print the prediction result
print("Predicted Sales Amount:", result['prediction'])

ValueError: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- Date
- Memo
- Num
- Type
